# Cross-sectional crypto momentum (top-K)

Same idea as the cross-asset / sector / country momentum notebooks (01, 02, 05), applied to a basket of liquid coins on yfinance. Each rebalance, rank coins by recent return and equal-weight the top K.

**Universe.** BTC, ETH, SOL, ADA, DOGE, XRP, BNB, LTC — the spot-USD pairs that have been continuously available on yfinance. The universe is intentionally small and survivorship-biased; treat results as upper bound.

**Signal.** 90-day total return excluding the most recent 7 days (skip the very-short reversal window). Weekly rebalance, equal-weight top K with K=3 by default.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.portfolio.long_only import fixed_weight_returns
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly

UNIVERSE = ["BTC-USD", "ETH-USD", "SOL-USD", "ADA-USD", "DOGE-USD", "XRP-USD", "BNB-USD", "LTC-USD"]
START = "2018-01-01"
END = None
LOOKBACK = 90          # calendar-day momentum window
SKIP = 7               # skip the last week to dodge short-term reversal
TOP_K = 3
REBAL = "W-FRI"
COSTS_BPS = 5.0
SLIPPAGE_BPS = 10.0

In [ ]:
prices = load_prices(UNIVERSE, start=START, end=END).dropna(how="all")
# Some pairs (SOL, BNB) start later; allow staggered availability and just rank what's listed.
prices = prices.ffill()
prices.tail()

In [ ]:
# Momentum: ret over LOOKBACK days ending SKIP days ago.
mom = prices.shift(SKIP) / prices.shift(SKIP + LOOKBACK) - 1

# At each date, rank only coins with non-NaN momentum (i.e. enough history).
ranks = mom.rank(axis=1, ascending=False, na_option="keep")
weights = (ranks <= TOP_K).astype(float).div(TOP_K)
weights = weights.where(mom.notna(), 0.0)
weights.tail()

In [ ]:
res = run_backtest(weights, prices, rebalance=REBAL, costs_bps=COSTS_BPS, slippage_bps=SLIPPAGE_BPS)

# Benchmarks: BTC-only buy-and-hold, and equal-weight-all buy-and-hold.
rets = prices.pct_change().fillna(0.0)
ew_all = fixed_weight_returns(rets, {t: 1.0 / len(UNIVERSE) for t in UNIVERSE})

streams = pd.DataFrame({
    f"top-{TOP_K} momentum": res.returns,
    "EW basket BH": ew_all,
    "BH BTC": rets["BTC-USD"],
}).dropna(how="all")
pd.DataFrame({name: pd.Series(summary(s)) for name, s in streams.items()})

In [ ]:
# How often does each coin show up in the basket? Concentration risk check.
(weights > 0).mean().sort_values(ascending=False).rename("pct_days_in_basket")

In [ ]:
annual_turnover = res.turnover.resample("YE").sum()
annual_costs = res.costs.resample("YE").sum()
pd.DataFrame({"turnover_per_year": annual_turnover, "cost_drag_per_year": annual_costs})

In [ ]:
equity_curve(streams)

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)

## Read

- Cross-sectional momentum on crypto historically rotated into whatever was running (ETH 2017, SOL 2021, etc.) — the question is whether it lets go of the loser quickly enough.
- This version is **always long top K** even during full-basket drawdowns. To survive bear markets, gate it on an absolute / trend filter (see notebook 08) or pair it with a cash sleeve.
- Universe is small and survivorship-biased. If results look great, run with a noisier coin (e.g. add SHIB / AVAX) and re-check.